# LeaseSight GPU-Accelerated Benchmark & Ablation Suite (Google Colab T4/V100)

This notebook packages the complete **LeaseSight** legal contract evaluation suite for maximum execution speed on GPU instances (Google Colab T4 / V100 / A100).

### Included Evaluations:
1. **SEC EDGAR 100-PDF Commercial Contracts Benchmark** (Top-10 Retrieval Depth, Multi-Pass Synonym Expansion).
2. **4-Variant Ablation Study** (Full LeaseSight Engine vs. Flat RAG, Dense-Only, and Sparse BM25-Only).
3. **CUAD 100-PDF Baseline Benchmark**.


In [ ]:
# Cell 1: Environment Setup & Dependency Installation
!pip install -q torch transformers sentence-transformers rank_bm25 pypdf PyPDF2 numpy

import os
import sys
import time
import json
import glob
import torch
from pathlib import Path

print("✓ Environment & dependencies initialized.")

In [ ]:
# Cell 2: GPU Verification & CUDA Device Setup
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_count = torch.cuda.device_count()
    print(f"✓ CUDA GPU Detected: {gpu_name} ({gpu_count} device(s) available)")
    print(f"  VRAM Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    device = "cuda"
else:
    print("⚠️ CUDA is not available. Falling back to CPU mode.")
    device = "cpu"

In [ ]:
# Cell 3: Dataset Preparation & Cache Purge
PROJECT_ROOT = Path(os.getcwd())
BENCHMARK_DIR = PROJECT_ROOT / "data" / "sec_edgar_test_100"

# Ensure clean cache state
for cdir in [PROJECT_ROOT / "data" / "json_maps", PROJECT_ROOT / "data" / "cache", PROJECT_ROOT / "data" / "temp"]:
    if cdir.exists():
        for item in cdir.iterdir():
            if item.is_file(): item.unlink()
    else:
        cdir.mkdir(parents=True, exist_ok=True)

pdf_count = len(glob.glob(str(BENCHMARK_DIR / "*.pdf"))) + len(glob.glob(str(BENCHMARK_DIR / "*.PDF")))
print(f"✓ Clean state verified. Discovered {pdf_count} benchmark contract PDFs in {BENCHMARK_DIR.name}")

In [ ]:
# Cell 4: CUDA-Accelerated Embedding & Hybrid RAG Benchmark Execution
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import numpy as np

print(f"Loading BAAI/bge-small-en-v1.5 embedding engine on device='{device}'...")
model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
print(f"✓ Embedding engine loaded on {device.upper()}.")

# Run evaluation script
!python scripts/run_sec_edgar_evaluation.py

In [ ]:
# Cell 5: Run Ablation Study & Display Markdown Results Summary
!python scripts/run_ablation_study.py

# Display summary report
report_path = Path("ABLATION_BENCHMARK_REPORT.md")
if report_path.exists():
    with open(report_path, "r", encoding="utf-8") as f:
        print(f.read())